In [1]:
# 1. IMPORT LIBRARY UTAMA
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score

print("--- Memulai Proses Data Preprocessing ---")

--- Memulai Proses Data Preprocessing ---


In [2]:
# 2. LOAD DATASET FINAL KELOMPOK ANDA
# Membaca data yang sudah memiliki label RUL dengan separator ';'
df = pd.read_csv('Dataset_Katup_Mesin_Final_Lengkap_Label.csv', sep=';')

In [3]:
# 3. PEMISAHAN FITUR INPUT (X) DAN LABEL TARGET (Y)
# Menggunakan 6 atribut sensor universal yang sudah dibersihkan ekstrem
X = df[['Suhu_Total_Kompresi_HPC', 'Suhu_Total_Turbin_LPT', 'Rasio_Tekanan_Mesin',
        'Tekanan_Statis_Ruang_Bakar', 'Rasio_Aliran_Bahan_Bakar', 'Kecepatan_Poros_Koreksi']]
y = df['Sisa_Umur_RUL']

In [4]:
# 4. SPLIT DATA (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# 5. NORMALISASI DATA (Standardization)
# Sangat penting untuk ANN agar bobot neuron tidak bias karena perbedaan satuan sensor
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Memulai Training Model ANN ---")

--- Memulai Training Model ANN ---


In [6]:
# 6. MEMBUAT ARSITEKTUR MODEL ANN REGRESI (MLP)
# Menggunakan 3 Hidden Layers (128, 64, 32 neuron) dengan aktivasi ReLU dan optimizer Adam
model_ann = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True # Mencegah overfitting
)

# Proses Training
import time
start_time = time.time()
model_ann.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

print(f"Training Selesai! Waktu Training: {training_time:.2f} detik")

Training Selesai! Waktu Training: 13.39 detik


In [7]:
# 7. EVALUASI MODEL (MENGHITUNG AKURASI & MAPE)
y_pred = model_ann.predict(X_test_scaled)

# Hitung MAPE (Maksimal 10% sesuai ketentuan dosen)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- HASIL EVALUASI MODEL ---")
print(f"Nilai MAPE (Error) : {mape:.2f}% (Target Dosen: Max 10%)")
print(f"Nilai R-Square (R2): {r2:.4f}")
print(f"Nilai RMSE         : {rmse:.2f}")


--- HASIL EVALUASI MODEL ---
Nilai MAPE (Error) : 17099657005150354.00% (Target Dosen: Max 10%)
Nilai R-Square (R2): 0.4799
Nilai RMSE         : 48.75


In [8]:
# 8. MENYIMPAN MODEL & SCALER (Format .pkl untuk di-deploy ke Streamlit)
with open('model_katup_ann.pkl', 'wb') as f_model:
    pickle.dump(model_ann, f_model)

with open('scaler_katup.pkl', 'wb') as f_scaler:
    pickle.dump(scaler, f_scaler)

print("\nModel dan Scaler sukses disimpan!")


Model dan Scaler sukses disimpan!
